# Notebook 2: RGB LED, PWM, and a Button

In Notebook 1 you drove a single LED fully on or fully off. This notebook
builds on that in three parts:

1. **RGB** — one LED module with three color channels (red, green, blue)
   that mix together, the same way a TV or phone screen makes every color
   out of just red/green/blue light.
2. **PWM (duty cycle)** — *how* gpiozero actually makes a channel look
   "half as bright" even though a GPIO pin can only ever be fully HIGH or
   fully LOW at any instant.
3. **Button → RGB integration** — reading a physical input (a push button)
   and using it to drive a physical output (the RGB LED's color), which is
   the general **INPUT → PROCESS → OUTPUT** pattern every interactive
   embedded program follows, including the finished robot car.

Like Notebook 1, this is still low-risk: LEDs and a push button, no motors.
Everything from Notebook 1 (BCM numbering, 3.3V logic, GND, resistors,
"wire it before you run it") still applies here and won't be re-explained
in full — skim back to Notebook 1 if any of that is fuzzy.

This notebook uses two already-implemented, already-tested project modules:
`src/hardware/rgb_led.py` and `src/hardware/button.py` (both verified on the
real Pi at the electrical level before this notebook was written). As in
Notebook 1, we build up from the raw `gpiozero` classes first, then show how
the project wraps them — and for Part 3, we write new *notebook-level*
logic that **composes** those two already-tested modules together, rather
than adding new code to `src/`. Combining tested building blocks at the
notebook layer, instead of growing `src/` for every one-off demo, is the
pattern this project follows throughout.

## Part 1: RGB color mixing

An RGB LED is really **three LEDs in one package** — a red one, a green
one, and a blue one — sharing a single common pin. Each of the three color
LEDs can be driven independently, and human eyes blend the three colors of
light together as if they were one color. This is the same idea behind
every pixel on every screen you've ever looked at: red + green + blue light,
mixed in different intensities, can produce (almost) any color.

Full brightness on all three channels at once looks white. All three off
looks off. Two channels on and one off gives you the secondary colors:

| Red | Green | Blue | Result |
|---|---|---|---|
| on | off | off | Red |
| off | on | off | Green |
| off | off | on | Blue |
| on | on | off | Yellow |
| off | on | on | Cyan |
| on | off | on | Magenta |
| on | on | on | White |
| off | off | off | Off |

This project's RGB LED module (`src/hardware/rgb_led.py`) is a **common
cathode** module (see `RGB_COMMON_CATHODE = True` in `src/config.py`) —
meaning the shared pin is GND, and each color channel lights up when its own
GPIO pin is driven HIGH. (The opposite kind, common *anode*, shares a 3.3V
pin instead and lights each channel on LOW — this module's code already
handles both cases via that one config flag, but you don't need to worry
about it since you're wiring a common-cathode module.)

### Wire your RGB LED

Four wires, one per color channel plus the shared ground. This is a 4-leg
RGB LED module (or four individual LEDs, if that's what you have) wired
exactly like the single LED in Notebook 1, just three times over sharing one
GND:

| Connect | To | BCM pin | Physical pin |
|---|---|---|---|
| Red channel → resistor (220-330Ω) → | GPIO22 | 22 | physical pin 15 |
| Green channel → resistor (220-330Ω) → | GPIO23 | 23 | physical pin 16 |
| Blue channel → resistor (220-330Ω) → | GPIO24 | 24 | physical pin 18 |
| Common pin (shared cathode) → | any GND pin | — | e.g. physical pin 14, near the others |

Just like Notebook 1's single LED, **each color channel needs its own
current-limiting resistor** (~220-330Ω) in series — three resistors total,
one per channel, not one shared resistor for all three. The common/shared
leg goes straight to GND with no resistor needed on that leg.

If you're using a pre-built 4-pin RGB LED module (common on a breadboard-
friendly breakout board) rather than a bare 4-leg LED, the module usually
has the resistors already built in — check its markings/datasheet; if so
you can skip adding your own resistors for that module specifically. If
you're wiring a bare RGB LED yourself, treat it exactly like three separate
single-color LEDs from Notebook 1 sharing one ground leg.

Double-check all three colors light up (dimly is fine) before moving on —
if one channel never lights no matter what color you set, recheck that
specific channel's wiring/resistor/pin first, the same troubleshooting order
as Notebook 1.

### Explanation

`gpiozero.RGBLED` is gpiozero's built-in class for exactly this kind of
3-channel-plus-common LED — it manages three PWM channels together and
lets you set a color as one `(red, green, blue)` tuple. Import it first.

In [ ]:
from gpiozero import RGBLED


### Expected result

No output, no error.

### Physical result

Nothing yet — no hardware touched.

### Explanation

Create the `RGBLED` object, telling it which BCM pin drives each channel.
`active_high=True` matches our common-cathode wiring (each channel lights on
HIGH) — see the color-mixing section above for why this flag exists.

In [ ]:
rgb = RGBLED(red=22, green=23, blue=24, active_high=True)


### Expected result

No output, no error. (A `GPIOPinInUse`-type error here usually means a
previous cell run still has GPIO22/23/24 open — restart the kernel and
re-run from the top.)

### Physical result

The LED should currently be off — `RGBLED` defaults to off (`(0, 0, 0)`)
when created.

### Explanation

`gpiozero.RGBLED`'s `.color` property takes a tuple of three floats, each
**0.0 to 1.0** (not 0-255) — one per red/green/blue channel. Setting it to
`(1, 0, 0)` means "red channel fully on, green and blue fully off".

In [ ]:
rgb.color = (1, 0, 0)  # full red


### Expected result

No output, no error.

### Physical result

The LED should turn solid red.

### Explanation

Try green and blue the same way, then a mixed color. `(1, 1, 0)` is red +
green with no blue — yellow.

In [ ]:
rgb.color = (0, 1, 0)  # full green


### Expected result

No output, no error.

### Physical result

The LED should turn solid green.

In [ ]:
rgb.color = (0, 0, 1)  # full blue


### Expected result

No output, no error.

### Physical result

The LED should turn solid blue.

In [ ]:
rgb.color = (1, 1, 0)  # red + green = yellow


### Expected result

No output, no error.

### Physical result

The LED should turn yellow (or a yellowish blend, depending on your LED's exact colors).

### Explanation

Turn it off, then release the pins with `.close()`, just like Notebook 1's
LED cleanup — same reasoning: release GPIO22/23/24 so they can be reused
cleanly by the next section.

In [ ]:
rgb.color = (0, 0, 0)
rgb.close()


### Expected result

No output, no error.

### Physical result

The LED turns off, then stays off (pins released).

### From raw `RGBLED` to the project's `rgb_led.py` module

Same pattern as Notebook 1: the raw `gpiozero.RGBLED` calls above are
correct, but `src/hardware/rgb_led.py` wraps them into small, named,
reusable functions so the rest of the project doesn't repeat this by hand
everywhere. Two things worth knowing about this module before using it:

- **0-255 color values, not 0.0-1.0.** `set_rgb(rgb, r, g, b)` takes each
  channel as an integer **0-255** — the same convention you may already
  know from CSS colors, Arduino tutorials, or paint programs — and divides
  by 255 internally before handing the result to `gpiozero.RGBLED`, which
  itself still wants 0.0-1.0. This module's docstring calls this out
  explicitly: if you ever drop down to the underlying `gpiozero.RGBLED`
  object directly, remember *it* wants 0.0-1.0, not 0-255.
- **Named color helpers.** Instead of remembering `(255, 0, 0)` is red,
  the module gives you `set_red(rgb)`, `set_green(rgb)`, `set_blue(rgb)`,
  `set_yellow(rgb)`, `set_cyan(rgb)`, `set_magenta(rgb)`, `set_white(rgb)`,
  and `set_off(rgb)` — one function per row of the color-mixing table above.

Pins still come from `src/config.py` (`RGB_RED_PIN`, `RGB_GREEN_PIN`,
`RGB_BLUE_PIN`), the same "one source of truth" pattern as `LED_PIN` in
Notebook 1.

In [ ]:
import sys
sys.path.insert(0, '../src')

from hardware.rgb_led import (
    get_rgb_led,
    set_rgb,
    set_red,
    set_green,
    set_blue,
    set_yellow,
    set_cyan,
    set_magenta,
    set_white,
    set_off,
    cleanup,
)

rgb = get_rgb_led()
print("RGB LED ready (pins from config.py).")


### Expected result

`RGB LED ready (pins from config.py).` printed, no errors.

### Physical result

The LED should be off (module default on creation, same as raw `RGBLED`).

### Explanation

Now step through all eight named colors, pausing half a second between each
so you can watch it change. This is the same list the module's own smoke
test uses, and matches exactly what QA verified electrically on the real
Pi for all 8 named colors.

In [ ]:
import time

named_colors = [
    ("RED", set_red),
    ("GREEN", set_green),
    ("BLUE", set_blue),
    ("YELLOW", set_yellow),
    ("CYAN", set_cyan),
    ("MAGENTA", set_magenta),
    ("WHITE", set_white),
    ("OFF", set_off),
]

for name, set_fn in named_colors:
    print(f"Setting color: {name}")
    set_fn(rgb)
    time.sleep(0.5)


### Expected result

Eight `Setting color: ...` lines printed in order, about 4 seconds total.

### Physical result

The LED should visibly cycle through red, green, blue, yellow, cyan,
magenta, white, then off, in that order, half a second on each.

### Explanation

You can also mix a custom color directly with `set_rgb(rgb, r, g, b)` using
0-255 values. Try a dim orange-ish mix (full red, partial green, no blue).

In [ ]:
set_rgb(rgb, 255, 100, 0)  # orange-ish


### Expected result

No output, no error.

### Physical result

The LED should show an orange/amber tone — mostly red with a bit of green,
no blue.

### Explanation

Leave the RGB LED **on and off()'d, but don't close it yet** — Part 3
reuses this exact same `rgb` object to drive colors from button presses. Just
turn it off for now.

In [ ]:
set_off(rgb)


### Expected result

No output, no error.

### Physical result

The LED turns off.

## Part 2: PWM and duty cycle

You may be wondering: a GPIO pin can only be fully HIGH (3.3V) or fully LOW
(0V) at any given instant — so how did `rgb.color = (1, 1, 0)` a moment ago
produce a *blended* yellow, and how could an LED ever look "half bright"
instead of just on or off?

The answer is **PWM: Pulse Width Modulation**. Instead of holding a pin at
one constant voltage, the pin is switched HIGH and LOW very rapidly — many
hundreds or thousands of times per second, far faster than your eye (or an
LED's response time) can follow. What your eye perceives as "brightness" is
actually the *average* voltage over each rapid on/off cycle.

**Duty cycle** is the percentage of each cycle spent HIGH:

| Duty cycle | Pattern (within one cycle) | Perceived brightness |
|---|---|---|
| 0% | LOW the whole time | Off |
| 25% | HIGH for 1/4 of the cycle, LOW for 3/4 | Dim |
| 50% | HIGH half the time, LOW half the time | Medium |
| 75% | HIGH for 3/4 of the cycle, LOW for 1/4 | Bright |
| 100% | HIGH the whole time | Full brightness (same as plain `.on()`) |

This is exactly what `gpiozero.RGBLED` was doing under the hood every time
you set a `.color` value that wasn't a plain 0 or 1 — and it's how
`set_rgb(rgb, 255, 100, 0)` a moment ago made green noticeably dimmer than
red without red and green being "the same brightness dialed differently" in
any way you had to manage by hand.

To see duty cycle in isolation, without the red/green/blue mixing on top,
we'll use `gpiozero.PWMLED` — a single-channel PWM-capable LED class — on
the same GPIO17 pin from Notebook 1. If your Notebook 1 LED (with its
resistor) is still wired to GPIO17/physical pin 11, you can reuse it here
with no rewiring. If you've since rewired that pin for something else,
rewire a single LED + resistor back to GPIO17/pin 9(GND)/pin 11 per
Notebook 1's instructions before running this section.

In [ ]:
from gpiozero import PWMLED


### Expected result

No output, no error.

### Physical result

Nothing yet.

### Explanation

Create a `PWMLED` on GPIO17 (same pin/wiring as Notebook 1's plain LED —
`PWMLED` just adds brightness control on top of the same on/off behavior).

In [ ]:
pwm_led = PWMLED(17)


### Expected result

No output, no error.

### Physical result

The LED should be off (default state).

### Explanation

`.value` on a `PWMLED` is the duty cycle as a float from 0.0 (0%, off) to
1.0 (100%, full brightness). Step through 0%, 25%, 50%, 75%, 100%, pausing a
second at each so you can compare brightness by eye.

In [ ]:
duty_cycles = [0.0, 0.25, 0.5, 0.75, 1.0]

for duty in duty_cycles:
    print(f"Duty cycle: {duty * 100:.0f}%")
    pwm_led.value = duty
    time.sleep(1)


### Expected result

Five lines printed (`Duty cycle: 0%` through `Duty cycle: 100%`), about 5
seconds total.

### Physical result

The LED should get visibly brighter at each step: off, dim, medium, bright,
full — a smooth ramp-up rather than a sudden jump, since your eye is
averaging each duty cycle's rapid on/off switching.

### Explanation

Cleanup: turn it off and release GPIO17.

In [ ]:
pwm_led.off()
pwm_led.close()


### Expected result

No output, no error.

### Physical result

The LED turns off; GPIO17 released.

This is exactly the mechanism `RGBLED` (and this project's `rgb_led.py`,
via `set_rgb`'s 0-255-to-0.0-1.0 conversion) relies on for every channel —
"color mixing" in Part 1 was really three independent PWM duty cycles
running at once, one per channel.

## Part 3: Reading a button, and driving the RGB LED from it

So far, every output (LED on/off, LED color) has been triggered by *you*
running a code cell. Now we introduce an **input**: a physical push button
that the Pi reads, and we'll use what it reads to decide what the RGB LED
should do.

This is the general shape of almost all interactive embedded/robotics code:

**INPUT** (read a sensor/button) → **PROCESS** (decide what that reading
means) → **OUTPUT** (act on that decision — light an LED, move a motor,
etc.)

Right now the "process" step will be simple (count button presses, map the
count to a color), but this exact same shape — read something, decide
something, act on something — is what will later read an ultrasonic
distance and decide whether to steer the robot, or read a camera frame and
decide what's in it. Learning to see code in these three stages now pays
off later.

### Wire your button

A simple 2-leg push button, wired between **GPIO27** and a **GND** pin —
**no external resistor needed**. The Pi's internal pull-up resistor handles
this (more on that below).

| Connect | To | BCM pin | Physical pin |
|---|---|---|---|
| One button leg → | GPIO27 | 27 | physical pin 13 |
| Other button leg → | any GND pin | — | e.g. physical pin 14, near pin 13 |

If your push button has 4 legs (common for breadboard tactile buttons),
that's normal — the 4 legs are really only 2 electrical connections (two
pairs, internally bridged in twos across the button's body). Use one leg
from each pair; if you're not sure which legs pair together, a
breadboard-straddling orientation (button straddling the center gap) almost
always puts each pair on opposite sides, and either leg within a pair works.

**How the "no resistor" part works:** `gpiozero.Button` enables the Pi's
**internal pull-up resistor** by default (`pull_up=True`), which quietly
holds GPIO27 at a HIGH (3.3V) reading whenever nothing else is pulling it
down. Pressing the button connects GPIO27 straight to GND, overpowering the
weak internal pull-up and pulling the reading LOW. So:

- Button **not** pressed → internal pull-up holds GPIO27 HIGH →
  `is_pressed` is `False`
- Button **pressed** → GPIO27 shorted to GND → `is_pressed` is `True`

Notice this is "backwards" from how you might first guess (pressed = LOW,
not pressed = HIGH) — that's normal for this very common pull-up wiring
style, and `src/hardware/button.py`'s docstring documents it for exactly
this reason. `gpiozero.Button` handles this inversion for you, so
`.is_pressed` already reads `True`/`False` the intuitive way (pressed =
`True`) regardless of the underlying voltage direction.

### Explanation

Import `Button` from `gpiozero` and create one on GPIO27, matching how you
wired it above.

In [ ]:
from gpiozero import Button

button = Button(27)
print("Button ready on GPIO27.")


### Expected result

`Button ready on GPIO27.` printed, no error.

### Physical result

Nothing visible yet — creating the object doesn't do anything by itself.

### Explanation

Read `.is_pressed` once, right now. Whatever the button's current physical
state is (pressed or not) at the moment this cell runs is what you'll see.

In [ ]:
print(button.is_pressed)


### Expected result

`False` if you're not touching the button as the cell runs, `True` if you
are actively holding it down when the cell executes.

### Physical result

None — this only reads the button's state, it doesn't change anything.

Try running this same cell again while physically holding the button down,
then again after releasing it, to see the value flip.

### Explanation

Release this raw `Button` object's pin before moving on. This matters here
more than it might seem: GPIO27 can only be held open by one `Button`/
`gpiozero` object at a time, so if you skip this and then try to create a
second `Button`/`get_button()` on GPIO27 below, it will raise a
"pin already in use" error.

In [ ]:
button.close()


### Expected result

No output, no error.

### Physical result

None — GPIO27 is simply released.

### Explanation

Now bring in the project's `button.py` module, which wraps this in
`get_button()` (reading `BUTTON_PIN` from `config.py`, same "one source of
truth" pattern as the LED pin). We'll use this version — `btn` — for the
rest of the notebook so the pin number stays centralized.

In [ ]:
from hardware.button import get_button

btn = get_button()
print("Project button module ready.")


### Expected result

`Project button module ready.` printed, no error.

### Physical result

None yet.

### Explanation — INPUT → PROCESS → OUTPUT, live

This is the integration exercise: **poll the button in a loop for a few
seconds, count each press, and use that count to decide the RGB LED's
color.** This loop is new code written here in the notebook — it isn't
added to `src/`, because it's a one-off demo *composing* the two
already-tested modules (`button.py` for input, `rgb_led.py` for output)
rather than a new reusable hardware capability of its own.

The mapping, following the spec for this notebook:

- 1st press → RED
- 2nd press → GREEN
- 3rd press → BLUE
- 4th press → YELLOW
- 5th press → OFF (and the count resets)

The loop below watches for the moment `is_pressed` transitions from `False`
to `True` (a "press edge") so each physical press counts exactly once,
rather than counting `True` repeatedly for as long as you hold the button
down. It runs for 20 seconds and then stops on its own — go press the
button up to several times while it's running.

In [ ]:
press_count = 0
last_state = False

color_for_count = {
    1: ("RED", set_red),
    2: ("GREEN", set_green),
    3: ("BLUE", set_blue),
    4: ("YELLOW", set_yellow),
}

print("Watching for button presses for 20 seconds. Go press the button!")
start = time.monotonic()

while time.monotonic() - start < 20:
    state = btn.is_pressed  # INPUT

    if state and not last_state:  # detected a new press (False -> True edge)
        press_count += 1
        if press_count > 5:
            press_count = 1  # wrap back around instead of stopping

        if press_count == 5:  # PROCESS
            print("Press 5: OFF")
            set_off(rgb)       # OUTPUT
        else:
            name, set_fn = color_for_count[press_count]  # PROCESS
            print(f"Press {press_count}: {name}")
            set_fn(rgb)                                   # OUTPUT

    last_state = state
    time.sleep(0.02)

print("Done watching for presses.")


### Expected result

`Watching for button presses for 20 seconds. Go press the button!`, then one
line per press (`Press 1: RED`, `Press 2: GREEN`, ...), then
`Done watching for presses.` after 20 seconds. If you press more than 5
times, the count wraps back to 1 (RED) instead of erroring or getting stuck.

### Physical result

Each press should immediately change the RGB LED: 1st press → red, 2nd →
green, 3rd → blue, 4th → yellow, 5th → off, then the pattern repeats from
red again. If nothing changes when you press, recheck the button wiring
(GPIO27 to GND, physical pins 13/14) before assuming the code is wrong —
the same troubleshooting order as earlier sections.

### Explanation

Cleanup: turn the LED off and release both the RGB LED's pins and the
button's pin.

In [ ]:
set_off(rgb)
cleanup(rgb)     # from hardware.rgb_led
btn.close()
print("RGB LED and button GPIO released.")


### Expected result

`RGB LED and button GPIO released.` printed, no error.

### Physical result

The LED turns off and stays off; the button no longer affects anything
since its pin has been released.

## Recap

- An RGB LED is three LEDs (red/green/blue) in one package, mixed by eye
  into other colors — this project's `src/hardware/rgb_led.py` wraps
  `gpiozero.RGBLED` with 0-255 color values and named helpers
  (`set_red`, `set_yellow`, etc.), all sourced from `RGB_RED_PIN` /
  `RGB_GREEN_PIN` / `RGB_BLUE_PIN` in `src/config.py`.
- **PWM (duty cycle)** is how a pin that can only be fully HIGH or fully LOW
  produces something that looks like a brightness in between — by switching
  rapidly and varying the fraction of time spent HIGH. This is what
  `gpiozero.RGBLED` and `PWMLED` are doing under the hood any time you set a
  value other than a plain 0 or 1.
- A push button wired to GPIO27 + GND, read with `gpiozero.Button`'s
  `.is_pressed` (via this project's `src/hardware/button.py`), relies on the
  Pi's internal pull-up resistor — no external resistor needed, and
  `is_pressed` already gives you the intuitive `True` = pressed reading.
- The button-driven color-cycling loop is the **INPUT → PROCESS → OUTPUT**
  pattern in miniature: read a sensor, decide what that reading means, act
  on an actuator. You'll see this same shape again, at larger scale, once
  the ultrasonic sensor and camera notebooks introduce inputs that decide
  how the robot drives.

## Exercises

**1. Create purple.**
Using `set_rgb(rgb, r, g, b)`, find a red/blue mix (green at or near 0)
that looks like purple/violet to you rather than plain magenta. (Hint: pure
magenta is `(255, 0, 255)` — try lowering red or blue somewhat unevenly and
see how the hue shifts.)

**2. Create orange.**
Similarly, using `set_rgb`, find an `(r, g, 0)` combination that reads as
orange rather than yellow. Compare it to the `(255, 100, 0)` example used
earlier in Part 1 — try a couple of different green values and note how the
perceived color shifts between "yellow-ish" and "red-ish" orange.

**3. Auto-cycle through colors.**
Write a loop that automatically steps through several colors (reuse the
`named_colors` list style from Part 1, or write your own list of `(name,
set_fn)` pairs) with `time.sleep()` between each, with no button involved —
a simple timed color-cycling "idle" pattern rather than the earlier
press-driven one.

**4. Extend the button pattern.**
Modify the Part 3 loop so a **long press** (held down for, say, over 1
second) does something different than a quick tap — for example, a long
press immediately jumps to OFF regardless of the current count, while a
quick tap advances through the color sequence as before. (Hint: you'll need
to record *when* `is_pressed` became `True`, using `time.monotonic()`, and
compare that timestamp to when it becomes `False` again to measure how long
it was held.)